In [1]:
import numpy as np
import pandas as pd

In [2]:
def performance(df: pd.DataFrame, col: str = "strat_ret", freq: int = 252):
    """
    Expects:
      - df[col]: strategy daily returns (including 0 when flat)
      - df['pos']: position series (0 when flat; >0 long; <0 short)  [optional but used]
      - df['delta_pos']: change in position to detect trade boundaries [optional but used]
    """
    r = df[col].dropna()


    ann_ret = (1 + r).prod()**(freq / max(len(r), 1)) - 1
    ann_vol = r.std() * np.sqrt(freq)
    sharpe  = 0.0 if ann_vol == 0 else ann_ret / ann_vol

    downside = r[r < 0]
    dd_vol   = downside.std() * np.sqrt(freq)
    sortino  = 0 if dd_vol == 0 else ann_ret / dd_vol


    equity   = (1 + r).cumprod()
    peak     = equity.cummax()
    drawdown = (equity / peak) - 1
    max_dd   = drawdown.min()


    win_rate_all     = (r > 0).mean()


    nonzero_mask     = r != 0
    win_rate_nonzero = (r[nonzero_mask] > 0).mean() if nonzero_mask.any() else np.nan


    if "pos" in df.columns:
        r_act = df.loc[df["pos"] != 0, col].dropna()
        win_rate_active = (r_act > 0).mean() if len(r_act) else np.nan
    else:
        win_rate_active = np.nan


    trade_rets = []
    n_trades   = 0
    if {"delta_pos", "pos"}.issubset(df.columns):
        in_pos = False
        acc = 0.0
        for _, row in df.iterrows():
            dp = row["delta_pos"]
            if (not in_pos) and (dp > 0):    # entry
                in_pos = True
                acc = 0.0
            if in_pos:
                acc += float(row[col])
            if in_pos and (dp < 0):          # exit
                trade_rets.append(acc)
                n_trades += 1
                in_pos = False

        if in_pos:
            trade_rets.append(acc)
            n_trades += 1

    trade_rets = np.array(trade_rets, dtype=float) if len(trade_rets) else np.array([])
    trade_win_rate     = float((trade_rets > 0).mean()) if trade_rets.size else np.nan
    avg_trade_return   = float(trade_rets.mean()) if trade_rets.size else np.nan
    median_trade_return= float(np.median(trade_rets)) if trade_rets.size else np.nan
    gross_win = trade_rets[trade_rets > 0].sum() if trade_rets.size else 0.0
    gross_loss= trade_rets[trade_rets < 0].sum() if trade_rets.size else 0.0
    profit_factor      = (gross_win / abs(gross_loss)) if gross_loss < 0 else (np.inf if gross_win > 0 else np.nan)

    turnover = df["delta_pos"].sum() / (len(df) / freq) if "delta_pos" in df.columns else np.nan

    return {
        "n_days":           int(len(r)),
        "ann_return":       float(ann_ret),
        "ann_vol":          float(ann_vol),
        "sharpe":           float(sharpe),
        "sortino":          float(sortino),
        "max_drawdown":     float(max_dd),

        # Daily win rates
        "win_rate":         float(win_rate_all),       
        "win_rate_nonzero": float(win_rate_nonzero),  
        "win_rate_active":  float(win_rate_active),     

        # Trade-level metrics
        "n_trades":         int(n_trades),
        "trade_win_rate":   float(trade_win_rate),
        "avg_trade_return": float(avg_trade_return) if not np.isnan(avg_trade_return) else np.nan,
        "median_trade_return": float(median_trade_return) if not np.isnan(median_trade_return) else np.nan,
        "profit_factor":    float(profit_factor) if np.isfinite(profit_factor) else (np.inf if profit_factor == np.inf else np.nan),

        # Activity
        "turnover_pa":      float(turnover),
    }